In [5]:
#!/usr/bin/env python3
"""
SMS / argot toxique  →  français standard — Extracteur de paires via Gemini (Colab).

Pipeline :
  1. Charge les 100 premiers textes de CyberBullyingAdo.parquet
  2. Filtre les entrées sans lettre latine
  3. Envoie chaque texte à Gemini avec un prompt-template strict
  4. Parse les réponses JSON, stocke les paires {source, target} + contexte
  5. Détecte la polysémie, produit un CSV et un JSON déduplicé
"""

import json
import re
import time
import pandas as pd
from google.colab import ai


PARQUET_PATH = "./CyberBullyingAdo.parquet"
N_TEXTS      = 100
OUTPUT_CSV   = "sms_to_french_pairs.csv"
OUTPUT_JSON  = "sms_to_french_pairs.json"
API_DELAY    = 0.5          # secondes entre chaque appel (anti rate-limit)


df = pd.read_parquet(PARQUET_PATH)
print(f"[info] Dataset complet : {len(df)} lignes")

df_subset = df.head(N_TEXTS).copy()
print(f"[info] Sous-ensemble  : {len(df_subset)} premières lignes")

# Précaution 1 : garder uniquement les textes contenant ≥ 1 lettre latine
_LATIN_RE = re.compile(
    r'[a-zA-ZàâäéèêëïîôöùûüÿçœæÀÂÄÉÈÊËÏÎÔÖÙÛÜŸÇŒÆ]'
)

def has_latin(text: str) -> bool:
    return isinstance(text, str) and bool(_LATIN_RE.search(text))

df_filtered = df_subset[df_subset['TEXT'].apply(has_latin)].copy()
n_removed = len(df_subset) - len(df_filtered)
print(f"[info] Après filtre latin : {len(df_filtered)} textes retenus "
      f"({n_removed} écarté{'s' if n_removed > 1 else ''})\n")


# ── Chaînes suspectes issues de l'analyse CamemBERT précédente ───────
#    (intégrées dans le prompt pour guider le LLM)
KNOWN_SUSPECTS = (
    "tkt, ptn, ntm, ftg, vrm, vrmt, tjrs, srx, dsl, jte, mtn, ouf, "
    "tqt, jsp, msg, bjr, wlh, fdp, tdc, jtm, nn, bh, xd, lvdm, cc, "
    "tktt, tktp, tsais, jss, mwa, lea, çca, psq, léa, 5eu, tai, sonb, "
    "cbvn, tfk, ftp, ltn, fgtg, irl, rsa, vpn, vmt, vbn, ojn, tjr, "
    "mvs, csc, uqi, oeeee, dégouteeee, bisoussssssssssssssssssssssssssss, "
    "mdmrmrr, bk, dz, pg, sl, ls, da, ds, cq"
)

PROMPT_TEMPLATE = """\
Tu es un expert en linguistique française, spécialisé dans l'argot des \
jeunes de 12-20 ans et dans le langage toxique du cyber-harcèlement en \
ligne (réseaux sociaux, SMS, messageries).

## TÂCHE
Analyse le texte ci-dessous. Identifie **chaque** mot ou groupe de mots \
correspondant à l'un de ces cas :
1. Mal orthographié volontairement / écriture SMS / abréviation \
   (ex : "rep" → "réponds", "vrm" → "vraiment", "bcp" → "beaucoup")
2. Insulte abrégée ou codée \
   (ex : "fdp" → "fils de pute", "ntm" → "nique ta mère", "tg" → "ta gueule")
3. Verlan ou argot absent d'un dictionnaire français standard \
   (ex : "teubé" → "bête", "zer" → "laser/bizarre selon contexte")
4. Chiffre(s) substitué(s) à des lettres \
   (ex : "2m1" → "demain", "3a0" → "zéro")
5. Mot tronqué \
   (ex : "tjrs" → "toujours", "mtn" → "maintenant", "srx" → "sérieux")
6. Émoticône textuelle : traduis par sa signification émotionnelle \
   (ex : "xd" → "rire", ":)" → "sourire", "^^" → "content")
7. Onomatopée ou mot allongé : donne la forme standard \
   (ex : "bisoussss" → "bisous", "trooooop" → "trop", "oeeee" → "oh")

## CHAÎNES SUSPECTES CONNUES (à rechercher en priorité)
{suspects}

## RÈGLES STRICTES
- Retourne **UNIQUEMENT** un tableau JSON valide. Aucun texte, aucun \
  commentaire, aucun markdown autour.
- Chaque élément : {{"source": "<mot exact du texte>", "target": "<forme standard>"}}.
- **NE traduis PAS** les mots existant dans un dictionnaire français \
  standard, même familiers : "ouais", "ok", "mec", "meuf", "truc", \
  "genre", "gars", "bof", "sympa", "mdr", "lol", "nul", "chiant", \
  "pote" sont valides — ne les corrige pas.
- **NE corrige PAS** la grammaire ni la syntaxe (accords, conjugaisons, \
  ponctuation). Corrige uniquement le **vocabulaire** SMS/abrégé/toxique.
- **Respecte le contexte** pour les mots polysémiques. \
  "sa" peut être "ça" ou le possessif "sa" : choisis selon la phrase.
- Si le texte ne contient AUCUN mot à corriger, retourne : []

## EXEMPLES IN-CONTEXT

Input: "t ou ptn tu rep jamais fdp"
Output: [{{"source": "t", "target": "t'es"}}, {{"source": "ptn", "target": "putain"}}, {{"source": "rep", "target": "réponds"}}, {{"source": "fdp", "target": "fils de pute"}}]

Input: "wlh tkt jvais le niquer srx"
Output: [{{"source": "wlh", "target": "wallah"}}, {{"source": "tkt", "target": "t'inquiète"}}, {{"source": "jvais", "target": "je vais"}}, {{"source": "srx", "target": "sérieux"}}]

Input: "Salut"
Output: []

## TEXTE À ANALYSER
\"\"\"{text}\"\"\"

## SORTIE JSON :"""


all_pairs: list[dict]   = []
failed_indices: list[int] = []
empty_count = 0

print("═" * 72)
print(f"  Traitement de {len(df_filtered)} textes via Gemini (Colab AI)")
print("═" * 72 + "\n")

for i, (row_idx, row) in enumerate(df_filtered.iterrows()):
    text = str(row['TEXT']).strip()
    if not text:
        continue

    # Construire le prompt (échapper les guillemets dans le texte)
    prompt = PROMPT_TEMPLATE.format(
        text=text.replace('"', '\\"'),
        suspects=KNOWN_SUSPECTS,
    )

    try:
        # ── Appel API Gemini Colab ────────────────────────────────────
        response = ai.generate_text(prompt)
        raw = response.strip()

        # ── Extraction du tableau JSON ────────────────────────────────
        # Le LLM peut entourer de ```json … ``` ou ajouter du texte
        json_match = re.search(r'\[.*?\]', raw, re.DOTALL)

        if json_match:
            pairs = json.loads(json_match.group(0))

            if not isinstance(pairs, list):
                raise ValueError("La réponse JSON n'est pas une liste")

            n_valid = 0
            for pair in pairs:
                if (isinstance(pair, dict)
                        and 'source' in pair
                        and 'target' in pair
                        and pair['source'].strip()
                        and pair['target'].strip()):
                    all_pairs.append({
                        'source':     pair['source'].strip().lower(),
                        'target':     pair['target'].strip().lower(),
                        'context':    text,
                        'text_index': int(row_idx),
                    })
                    n_valid += 1

            if n_valid:
                print(f"  [{i+1:3d}/{len(df_filtered)}] ✓ {n_valid:2d} paire(s)  "
                      f"│ {text[:60]}{'…' if len(text)>60 else ''}")
            else:
                empty_count += 1
                print(f"  [{i+1:3d}/{len(df_filtered)}] · aucun argot  "
                      f"│ {text[:60]}{'…' if len(text)>60 else ''}")

        elif '[]' in raw:
            empty_count += 1
            print(f"  [{i+1:3d}/{len(df_filtered)}] · aucun argot  "
                  f"│ {text[:60]}{'…' if len(text)>60 else ''}")
        else:
            print(f"  [{i+1:3d}/{len(df_filtered)}] ⚠ pas de JSON  "
                  f"│ réponse brute : {raw[:80]}")
            failed_indices.append(row_idx)

    except json.JSONDecodeError as exc:
        print(f"  [{i+1:3d}/{len(df_filtered)}] ✗ JSON invalide : {exc}")
        failed_indices.append(row_idx)
    except Exception as exc:
        print(f"  [{i+1:3d}/{len(df_filtered)}] ✗ Erreur API : {exc}")
        failed_indices.append(row_idx)

    time.sleep(API_DELAY)

print(f"\n{'═' * 72}")
print("  RÉSULTATS")
print("═" * 72)

pairs_df = pd.DataFrame(all_pairs)
n_pairs   = len(pairs_df)
n_unique  = pairs_df['source'].nunique() if n_pairs else 0

print(f"\n  Paires extraites       : {n_pairs}")
print(f"  Termes source uniques  : {n_unique}")
print(f"  Textes sans argot      : {empty_count}")
print(f"  Échecs (parse / API)   : {len(failed_indices)}")

if n_pairs == 0:
    print("\n  ⚠ Aucune paire n'a été extraite — vérifier les appels API.")
    raise SystemExit

# ── Précaution 2 : Détection de polysémie ────────────────────────────
polysemy = (
    pairs_df
    .groupby('source')['target']
    .apply(lambda s: sorted(set(s)))
    .reset_index()
)
polysemy['n_targets'] = polysemy['target'].apply(len)
multi = polysemy[polysemy['n_targets'] > 1].sort_values(
    'n_targets', ascending=False
)

if len(multi):
    print(f"\n  ⚠ Termes polysémiques ({len(multi)} source(s) → cibles multiples) :")
    for _, r in multi.iterrows():
        print(f"    '{r['source']}' → {r['target']}")

# ── Table de fréquences ──────────────────────────────────────────────
freq = (
    pairs_df
    .groupby(['source', 'target'])
    .agg(count=('source', 'size'))
    .reset_index()
    .sort_values('count', ascending=False)
    .reset_index(drop=True)
)

print(f"\n  Top-30 des mappings les plus fréquents :\n")
with pd.option_context('display.max_colwidth', 40, 'display.width', 120):
    print(freq.head(30).to_string(index=False))


# ── CSV : toutes les paires avec contexte ─────────────────────────────
pairs_df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
print(f"\n[saved] {OUTPUT_CSV}  ({n_pairs} lignes)")

# ── JSON : dictionnaire dédupliqué avec exemples de contexte ──────────
dict_entries = []
for src in pairs_df['source'].unique():
    sub = pairs_df[pairs_df['source'] == src]
    targets_agg = (
        sub.groupby('target')
        .agg(
            frequency=('target', 'size'),
            example_contexts=('context', lambda x: list(x.unique())[:3]),
        )
        .reset_index()
    )
    for _, t_row in targets_agg.iterrows():
        dict_entries.append({
            'source':           src,
            'target':           t_row['target'],
            'frequency':        int(t_row['frequency']),
            'example_contexts': t_row['example_contexts'],
        })

with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
    json.dump(dict_entries, f, ensure_ascii=False, indent=2)
print(f"[saved] {OUTPUT_JSON}  ({len(dict_entries)} entrées)")

print(f"\n{'═' * 72}")
print("  RÉSUMÉ")
print("═" * 72)
print(f"  • {len(df_filtered)} textes traités")
print(f"  • {n_pairs} paires (source → target) extraites")
print(f"  • {n_unique} termes source uniques")
print(f"  • {len(multi)} termes polysémiques détectés")
print(f"  • Fichiers : {OUTPUT_CSV}, {OUTPUT_JSON}")
print(f"\n✓  Pipeline terminé.\n")

[info] Dataset complet : 5608 lignes
[info] Sous-ensemble  : 100 premières lignes
[info] Après filtre latin : 99 textes retenus (1 écarté)

════════════════════════════════════════════════════════════════════════
  Traitement de 99 textes via Gemini (Colab AI)
════════════════════════════════════════════════════════════════════════

  [  1/99] ✓  2 paire(s)  │ WESH silvia pk tu vien et tu fou la merde dans les couple
  [  2/99] ✓  1 paire(s)  │ grave tu viens et tu pécho d'autre gars
  [  3/99] · aucun argot  │ tu parle de quoi c'est lucas qui a foutu la merde
  [  4/99] ✓  1 paire(s)  │ vas y t'es tellement une pute que tu irai draguer meme les m…
  [  5/99] ✓  1 paire(s)  │ nan mais les gars calmez vous quoi si lucas c'est lui la put…
  [  6/99] ✓  4 paire(s)  │ Lucas c'est mon meilleur je le connais tu lui a promis des n…
  [  7/99] ✓  2 paire(s)  │ Lucas il ma di qu'il voulait pas rester avecc sophie
  [  8/99] ✓  1 paire(s)  │ mon cul:@
  [  9/99] ✓  1 paire(s)  │ aller niquez vos